<a href="https://colab.research.google.com/github/anjaliisharmaa27-byte/Uber-Supply-Demand-gap-analysis/blob/main/Uber_SQL(New1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import sqlite3

# Load CLEANED Excel file from GitHub
url = 'https://raw.githubusercontent.com/anjaliisharmaa27-byte/Uber-Supply-Demand-gap-analysis/refs/heads/main/Updated_uber-data.xlsx'

df = pd.read_excel(url)

# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Create database
conn = sqlite3.connect('uber.db')
df.to_sql('trips', conn, if_exists='replace', index=False)

print("✅ Database ready!")
print(df.columns.tolist())

✅ Database ready!
['Request_id', 'Pickup_point', 'Driver_id', 'Status', 'Request_timestamp', 'Drop_timestamp', 'Hour', 'Time_of_Day']


In [20]:
print(df.columns.tolist())

['Request_id', 'Pickup_point', 'Driver_id', 'Status', 'Request_timestamp', 'Drop_timestamp', 'Hour', 'Time_of_Day']


In [21]:
# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Create SQLite database
conn = sqlite3.connect('uber.db')
df.to_sql('trips', conn, if_exists='replace', index=False)

# Save database to Drive (safe forever!)
import shutil
shutil.copy('uber.db', '/content/drive/MyDrive/uber.db')

print("✅ Database ready and saved to Drive!")

✅ Database ready and saved to Drive!


In [22]:
def run_query(sql, title=""):
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print(f"{'='*50}")
    display(result)
    return result

In [23]:
run_query("""
    SELECT
        Status,
        COUNT(*) AS Total_Trips
    FROM trips
    GROUP BY Status
    ORDER BY Total_Trips DESC
""", title="Query 1 - Trip Count by Status");


  Query 1 - Trip Count by Status


,Status,Total_Trips
0,Trip Completed,2831
1,No Cars Available,2650
2,Cancelled,1264


In [24]:
 run_query("""
    SELECT
        Hour,
        COUNT(*) AS Unfulfilled_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Hour
    ORDER BY Hour
""", title="Query 2 - Unfulfilled Requests Per Hour");


  Query 2 - Unfulfilled Requests Per Hour


,Hour,Unfulfilled_Requests
0,0,59
1,1,60
2,2,62
3,3,58
4,4,125
5,5,260
6,6,231
7,7,232
8,8,268
9,9,258


In [25]:
run_query("""
    SELECT
        Time_of_Day,
        COUNT(*) AS Unfulfilled_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Time_of_Day
    ORDER BY Unfulfilled_Requests DESC
""", title="Query 3 - Supply Demand Gap by Time of Day");


  Query 3 - Supply Demand Gap by Time of Day


,Time_of_Day,Unfulfilled_Requests
0,Evening,1251
1,Early Morning,991
2,Late Night,548
3,Morning,441
4,Night,364
5,Afternoon,319


In [26]:
run_query("""
    SELECT
        Pickup_point,
        Status,
        COUNT(*) AS Count
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Pickup_point, Status
    ORDER BY Pickup_point, Count DESC
""", title="Query 4 - Airport vs City Cancelled and No Cars");


  Query 4 - Airport vs City Cancelled and No Cars


,Pickup_point,Status,Count
0,Airport,No Cars Available,1713
1,Airport,Cancelled,198
2,City,Cancelled,1066
3,City,No Cars Available,937


In [27]:
run_query("""
    SELECT
        Driver_id,
        COUNT(*) AS Cancellations
    FROM trips
    WHERE Status = 'Cancelled'
        AND Driver_id IS NOT NULL
    GROUP BY Driver_id
    ORDER BY Cancellations DESC
    LIMIT 10
""", title="Query 5 - Top 10 Drivers by Cancellations");


  Query 5 - Top 10 Drivers by Cancellations


,Driver_id,Cancellations
0,84.0,12
1,54.0,11
2,206.0,10
3,142.0,10
4,267.0,9
5,210.0,9
6,166.0,9
7,138.0,9
8,114.0,9
9,27.0,9


In [28]:
run_query("""
    SELECT
        Pickup_point,
        COUNT(*) AS Total_Requests,
        SUM(CASE WHEN Status = 'Trip Completed' THEN 1 ELSE 0 END) AS Completed,
        ROUND(
            100.0 * SUM(CASE WHEN Status = 'Trip Completed' THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS Completion_Rate_Pct
    FROM trips
    GROUP BY Pickup_point
""", title="Query 6 - Completion Rate Airport vs City");


  Query 6 - Completion Rate Airport vs City


,Pickup_point,Total_Requests,Completed,Completion_Rate_Pct
0,Airport,3238,1327,41.0
1,City,3507,1504,42.9


In [29]:
run_query("""
    SELECT
        Pickup_point,
        Time_of_Day,
        COUNT(*) AS Problem_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Pickup_point, Time_of_Day
    ORDER BY Problem_Requests DESC
    LIMIT 10
""", title="Query 7 - Worst Pickup Point and Time Slot Combos");


  Query 7 - Worst Pickup Point and Time Slot Combos


,Pickup_point,Time_of_Day,Problem_Requests
0,Airport,Evening,1145
1,City,Early Morning,962
2,Airport,Late Night,421
3,City,Morning,389
4,City,Night,214
5,City,Afternoon,205
6,Airport,Night,150
7,City,Late Night,127
8,Airport,Afternoon,114
9,City,Evening,106
